In [8]:
import pandas as pd
import sqlite3

In [10]:
con = sqlite3.connect("datalake.db")
cur = con.cursor()


In [ ]:
# bronze_pessoas
cur.execute("""
    CREATE TABLE IF NOT EXISTS bronze_pessoas (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome TEXT,
            idade INTEGER,
            salario REAL
            )
""")


In [14]:
# bronze_vendas
cur.execute("""
    CREATE TABLE IF NOT EXISTS bronze_vendas (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            vendedor TEXT,
            valor REAL,
            data TEXT
            )
""")

In [41]:

con.commit()

In [23]:
cur.execute("""DELETE FROM bronze_pessoas""")

cur.executemany("""
    INSERT INTO bronze_pessoas (nome, idade, salario) VALUES (?, ?, ?)
""", [('João', 25, 2000.00), 
      ('Maria', 30, 3000.00), 
      ('José', 35, 4000.00),
      ('Ana', 40, 5000.00),
      ('Pedro', 38, 14000.00)      
      ])

In [27]:
cur.execute("""DELETE FROM bronze_vendas""")

cur.executemany("""
    INSERT INTO bronze_vendas (vendedor, valor, data) VALUES (?, ?, ?)
""", [('João', 1000.00, '2026-01-01'), 
      ('Maria', 2000.00, '2026-01-02'), 
      ('José', 3000.00, '2026-01-03'),
      ('Ana', 7000.00, '2026-01-04'),
      ('Pedro', 4000.00, '2026-01-05')      
      ])

In [28]:
cur.execute("""SELECT * FROM bronze_pessoas""")
resultado = cur.fetchall()
for linha in resultado:
    print(linha)

cur.execute("""SELECT * FROM bronze_vendas""")
resultado = cur.fetchall()
for linha in resultado:
    print(linha)

(11, 'João', 25, 2000.0)
(12, 'Maria', 30, 3000.0)
(13, 'José', 35, 4000.0)
(14, 'Ana', 40, 5000.0)
(15, 'Pedro', 38, 14000.0)
(11, 'João', 1000.0, '2026-01-01')
(12, 'Maria', 2000.0, '2026-01-02')
(13, 'José', 3000.0, '2026-01-03')
(14, 'Ana', 7000.0, '2026-01-04')
(15, 'Pedro', 4000.0, '2026-01-05')


# CAMADA SILVER

In [30]:
df_pessoas = pd.read_sql_query("SELECT * FROM bronze_pessoas", con)
print(df_pessoas)

   id   nome  idade  salario
0  11   João     25   2000.0
1  12  Maria     30   3000.0
2  13   José     35   4000.0
3  14    Ana     40   5000.0
4  15  Pedro     38  14000.0


In [32]:
df_vendas = pd.read_sql_query("SELECT * FROM bronze_vendas", con)
print(df_vendas)

   id vendedor   valor        data
0  11     João  1000.0  2026-01-01
1  12    Maria  2000.0  2026-01-02
2  13     José  3000.0  2026-01-03
3  14      Ana  7000.0  2026-01-04
4  15    Pedro  4000.0  2026-01-05


In [35]:
df_pessoas["faixa_etaria"] = df_pessoas["idade"].apply(
    lambda x : "Jovem" if x < 30 else "Adulto" if x <= 39 else "Senior"
)
print(df_pessoas)

   id   nome  idade  salario faixa_etaria
0  11   João     25   2000.0        Jovem
1  12  Maria     30   3000.0       Adulto
2  13   José     35   4000.0       Adulto
3  14    Ana     40   5000.0       Senior
4  15  Pedro     38  14000.0       Adulto


In [37]:
df_vendas = df_vendas[df_vendas["valor"]>2000]
print(df_vendas)

   id vendedor   valor        data
2  13     José  3000.0  2026-01-03
3  14      Ana  7000.0  2026-01-04
4  15    Pedro  4000.0  2026-01-05


In [39]:
cur.execute("""
    CREATE TABLE IF NOT EXISTS silver_pessoas (
            id INTEGER PRIMARY KEY,
            nome TEXT,
            idade INTEGER,
            salario REAL,
            faixa_etaria TEXT
            )
""")

In [40]:
cur.execute("""
    CREATE TABLE IF NOT EXISTS silver_vendas (
            id INTEGER PRIMARY KEY,
            vendedor TEXT,
            valor REAL,
            data TEXT
            )
""")

In [44]:
df_pessoas.to_sql("silver_pessoas", con, if_exists="replace", index=False)

5

In [46]:
df_vendas.to_sql("silver_vendas", con, if_exists="replace", index=False)

3

In [48]:
con.commit()

In [49]:
cur.execute("""SELECT * FROM silver_pessoas""")
resultado = cur.fetchall()
for linha in resultado:
    print(linha)


(11, 'João', 25, 2000.0, 'Jovem')
(12, 'Maria', 30, 3000.0, 'Adulto')
(13, 'José', 35, 4000.0, 'Adulto')
(14, 'Ana', 40, 5000.0, 'Senior')
(15, 'Pedro', 38, 14000.0, 'Adulto')


In [50]:
cur.execute("""SELECT * FROM silver_vendas""")
resultado = cur.fetchall()
for linha in resultado:
    print(linha)

(13, 'José', 3000.0, '2026-01-03')
(14, 'Ana', 7000.0, '2026-01-04')
(15, 'Pedro', 4000.0, '2026-01-05')


# CAMADA GOLD

In [53]:
cur.execute("""
    CREATE TABLE IF NOT EXISTS dim_pessoas (
            id INTEGER PRIMARY KEY,
            nome TEXT,
            faixa_etaria TEXT
            )
""")

con.commit()

In [55]:
cur.execute("""
    CREATE TABLE IF NOT EXISTS fat_salarios (
            id INTEGER PRIMARY KEY,
            pessoa_id INTEGER,
            salario REAL,
            FOREIGN KEY (pessoa_id) REFERENCES dim_pessoas(id)
        )
""")

con.commit()

In [57]:
cur.execute("""
    CREATE TABLE IF NOT EXISTS fat_vendas (
            id INTEGER PRIMARY KEY,
            pessoa_id INTEGER,
            total_vendas REAL,
            FOREIGN KEY (pessoa_id) REFERENCES dim_pessoas(id)
            )
""")

con.commit()

In [59]:
dim_pessoas = df_pessoas[["id", "nome", "faixa_etaria"]]
print(dim_pessoas)

   id   nome faixa_etaria
0  11   João        Jovem
1  12  Maria       Adulto
2  13   José       Adulto
3  14    Ana       Senior
4  15  Pedro       Adulto


In [61]:
fat_salarios = df_pessoas[["id", "salario"]].rename(columns={"id":"pessoa_id"})


In [64]:
fat_vendas = (df_vendas
              .groupby("vendedor")
              .agg({"valor":"sum"})
              .reset_index()
              .rename(columns={"vendedor":"nome", "valor":"total_vendas"})
              .merge(dim_pessoas, on="nome")[["id", "total_vendas"]]
              .rename(columns={"id":"pessoa_id"})
              )

In [65]:
dim_pessoas.to_sql("dim_pessoas", con, if_exists="replace", index=False)

5

In [66]:
fat_salarios.to_sql("fat_salarios", con, if_exists="replace", index=False)

5

In [68]:
fat_vendas.to_sql("fat_vendas", con, if_exists="replace", index=False)

3

In [71]:
df_dim_pessoas = pd.read_sql_query("SELECT * FROM dim_pessoas", con)
print(df_dim_pessoas)

   id   nome faixa_etaria
0  11   João        Jovem
1  12  Maria       Adulto
2  13   José       Adulto
3  14    Ana       Senior
4  15  Pedro       Adulto


In [73]:
df_fat_salarios = pd.read_sql_query("SELECT * FROM fat_salarios", con)
print(df_fat_salarios)

   pessoa_id  salario
0         11   2000.0
1         12   3000.0
2         13   4000.0
3         14   5000.0
4         15  14000.0


In [75]:
df_fat_vendas = pd.read_sql_query("SELECT * FROM fat_vendas", con)
print(df_fat_vendas)

   pessoa_id  total_vendas
0         14        7000.0
1         13        3000.0
2         15        4000.0


In [76]:
con.commit()
con.close()